In [2]:
import sys
import os
import pandas as pd
import prophet

sys.path.append(os.path.abspath(".."))

from Pipeline.DFLoader import DFLoader
from Pipeline.PipelienFactory import PipelineFactory

ModuleNotFoundError: No module named 'prophet'

In [7]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\Anwender\.pyenv\pyenv-win\versions\3.11.3\python.exe
3.11.3 (tags/v3.11.3:f3909b8, Apr  4 2023, 23:49:59) [MSC v.1934 64 bit (AMD64)]


In [19]:
df = pd.read_csv("../../files/data/lfs/UWB/data_tables/Station_784_2018-01-01--2025-12-31.csv")


print(df.head())
print(df.columns)

            date_start     5     4      1     3
0  2018-01-01 00:00:00  80.0  16.0  585.0   6.0
1  2018-01-01 01:00:00  28.0   3.0   98.0  54.0
2  2018-01-01 02:00:00  10.0   3.0   22.0  73.0
3  2018-01-01 03:00:00  14.0   3.0   20.0  68.0
4  2018-01-01 04:00:00  18.0   6.0   20.0  62.0
Index(['date_start', '5', '4', '1', '3'], dtype='object')


In [20]:
df['date_start'] = pd.to_datetime(df['date_start'])
df = df.sort_values('date_start').reset_index(drop=True)

In [21]:
df = df.rename(columns={'1': 'pm10'})

In [27]:
df['pm10'].describe()

count    68316.000000
mean        15.918116
std         12.188550
min          2.000000
25%          8.000000
50%         13.000000
75%         20.000000
max        762.000000
Name: pm10, dtype: float64

In [25]:
print(df[['date_start', 'pm10']].head())
print(df.dtypes)

           date_start   pm10
0 2018-01-01 00:00:00  585.0
1 2018-01-01 01:00:00   98.0
2 2018-01-01 02:00:00   22.0
3 2018-01-01 03:00:00   20.0
4 2018-01-01 04:00:00   20.0
date_start    datetime64[ns]
5                    float64
4                    float64
pm10                 float64
3                    float64
dtype: object


In [28]:
df_daily = df.resample('D', on='date_start').mean()

In [29]:
df_daily.head()
df_daily.shape

(2922, 4)

In [30]:
print(df_daily.head())
print(df_daily.columns)

                    5         4       pm10          3
date_start                                           
2018-01-01  23.375000  4.875000  38.708333  54.250000
2018-01-02  34.954545  3.681818  17.909091  23.545455
2018-01-03  17.454545  3.227273   5.625000  56.818182
2018-01-04  27.800000  3.952381  18.458333  43.904762
2018-01-05  28.625000  3.583333  10.625000  41.708333
Index(['5', '4', 'pm10', '3'], dtype='object')


In [31]:
df_daily['target'] = df_daily['pm10'].shift(-1)

print(df_daily[['pm10', 'target']].head())
print(df_daily[['pm10', 'target']].tail())

                 pm10     target
date_start                      
2018-01-01  38.708333  17.909091
2018-01-02  17.909091   5.625000
2018-01-03   5.625000  18.458333
2018-01-04  18.458333  10.625000
2018-01-05  10.625000  14.000000
                 pm10     target
date_start                      
2025-12-27  17.416667  13.791667
2025-12-28  13.791667  16.041667
2025-12-29  16.041667   9.833333
2025-12-30   9.833333  11.833333
2025-12-31  11.833333        NaN


In [33]:

print(os.getcwd())

c:\Users\Anwender\bootcamp\week13\Capstone\python\notebooks


In [39]:
# DF Loader testen

df = DFLoader.load_daily_df()

print(type(df))
print(df.shape)
print(df.head())
print(df.columns)

<class 'pandas.core.frame.DataFrame'>
(2922, 14)
                 PM10  precipitation  pressure_msl  sunshine  temperature  \
2018-01-01   9.916667       0.016667    747.425000  2.125000     5.041667   
2018-01-02  15.750000       0.000000    753.941667  0.125000     3.737500   
2018-01-03   6.458333       1.137500    735.866667  2.125000     4.529167   
2018-01-04  16.833333       0.220833    743.020833  0.041667     4.483333   
2018-01-05  12.250000       0.050000    744.641667  0.000000     3.991667   

            cloud_cover  dew_point  relative_humidity    visibility     solar  \
2018-01-01    60.208333   2.616667          60.000000  27291.666667  0.021208   
2018-01-02    67.500000   2.691667          68.041667  16625.000000  0.014333   
2018-01-03    69.208333   3.262500          66.750000  13270.833333  0.011667   
2018-01-04    70.166667   3.729167          69.958333   7583.333333  0.010833   
2018-01-05    66.000000   2.579167          65.875000  17208.333333  0.004292   

 

In [50]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureDropTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop, errors="ignore")

In [1]:
# Nächster Test

from Pipeline.PipelienFactory import PipelineFactory

main_pipe = PipelineFactory.create_main_pipeline()
df_trans = main_pipe.fit_transform(df)

print(type(df_trans))
print(df_trans.shape)

try:
    print(df_trans.head())
except:
    print(df_trans[:5])

ModuleNotFoundError: No module named 'Pipeline'

In [43]:
print(os.listdir("../Pipeline"))

['Decomposer', 'DFLoader.py', 'PipelienFactory.py', 'Regressor', 'Transformers', 'Wrapper', '__pycache__']


In [49]:
# Ruhekn Utils finden
print(os.listdir(".."))
print(os.listdir("../Pipeline"))

['DataAquisition', 'Evaluation', 'notebooks', 'Pipeline', 'Utils']
['Decomposer', 'DFLoader.py', 'PipelienFactory.py', 'Regressor', 'Transformers', 'Wrapper', '__pycache__']


In [46]:
for root, dirs, files in os.walk(".."):
    for f in files:
        if "ruhken" in f.lower():
            print(os.path.join(root, f))